In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert ".venv" in sys.executable, "Wrong kernel! Switch to 'Python (f1-tyre-strategy)' (top right)."
PROJECT_ROOT = Path.cwd().parent
assert PROJECT_ROOT.name == "f1-tyre-strategy-simulator", "Notebook isn't in the notebooks/ folder!"

RACES_DIR = PROJECT_ROOT / "data" / "raw" / "races"
QUALI_DIR = PROJECT_ROOT / "data" / "raw" / "quali"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DRY = ["SOFT", "MEDIUM", "HARD"]
WET = ["INTERMEDIATE", "WET"]
print("✅ Kernel and paths OK")

✅ Kernel and paths OK


In [2]:
laps = pd.concat([pd.read_parquet(f) for f in sorted(RACES_DIR.glob("*.parquet"))], ignore_index=True)
quali = pd.concat([pd.read_parquet(f) for f in sorted(QUALI_DIR.glob("*.parquet"))], ignore_index=True)

n_races = laps.groupby(["Year", "Round"]).ngroups
n_quali = quali.groupby(["Year", "Round"]).ngroups
print(f"Race laps: {laps.shape[0]:,} rows × {laps.shape[1]} columns | races: {n_races}")
print(f"Qualifying: {quali.shape[0]:,} driver rows | sessions: {n_quali}")

assert n_races == 92 and n_quali == 92, "Expected 92 races and 92 qualifying sessions"
print("✅ Both sources complete")

Race laps: 101,291 rows × 30 columns | races: 92
Qualifying: 1,831 driver rows | sessions: 92
✅ Both sources complete


In [3]:
# FastF1 renamed Miami's location in 2025; map it back so it's one circuit
CIRCUIT_FIXES = {"Miami Gardens": "Miami"}
laps["Circuit"] = laps["Location"].replace(CIRCUIT_FIXES)

races = laps.drop_duplicates(["Year", "Round"])[["Year", "Round", "EventName", "Circuit"]]
counts = races["Circuit"].value_counts()
print("Races per circuit:")
print(counts.to_string())

assert races["Circuit"].nunique() == 25, f"Expected 25 circuits, found {races['Circuit'].nunique()}"
assert counts["Miami"] == 4, "Miami should have 4 races (2022–2025)"
print("\n✅ 25 circuits; Miami now has 4 races")

Races per circuit:
Circuit
Sakhir               4
Jeddah               4
Yas Island           4
São Paulo            4
Mexico City          4
Austin               4
Suzuka               4
Marina Bay           4
Monza                4
Zandvoort            4
Spa-Francorchamps    4
Budapest             4
Spielberg            4
Silverstone          4
Montréal             4
Baku                 4
Monaco               4
Barcelona            4
Miami                4
Melbourne            4
Imola                3
Lusail               3
Las Vegas            3
Shanghai             2
Le Castellet         1

✅ 25 circuits; Miami now has 4 races


In [4]:
quali_cols = ["Year", "Round", "Driver", "BestQualiLapSeconds", "BestQualiCompound",
              "QualiWetTyreShare", "QualiRainfall"]

# One row per driver per race, with the results-table time from the race files
drivers = laps.drop_duplicates(["Year", "Round", "Driver"])[
    ["Year", "Round", "EventName", "Circuit", "Driver", "Team", "BestQualiSeconds"]
]
check = drivers.merge(quali[quali_cols], on=["Year", "Round", "Driver"], how="left")

both = check.dropna(subset=["BestQualiSeconds", "BestQualiLapSeconds"]).copy()
both["Diff"] = (both["BestQualiLapSeconds"] - both["BestQualiSeconds"]).round(3)

print(f"Race drivers: {len(check):,} | with both qualifying sources: {len(both):,}")
print(f"Sources identical (within 0.01 s): {(both['Diff'].abs() <= 0.01).mean():.1%}")
print(f"Differ by more than 0.1 s: {(both['Diff'].abs() > 0.1).sum()}")
print(both.loc[both["Diff"].abs() > 0.1,
               ["Year", "EventName", "Driver", "BestQualiSeconds", "BestQualiLapSeconds", "Diff"]]
      .sort_values("Diff").head(15).to_string(index=False))

print("\nOnly the results-table time:", (check["BestQualiSeconds"].notna() & check["BestQualiLapSeconds"].isna()).sum())
print("Only the lap-data time:     ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].notna()).sum())
print("Neither:                    ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].isna()).sum())

Race drivers: 1,827 | with both qualifying sources: 1,789
Sources identical (within 0.01 s): 97.1%
Differ by more than 0.1 s: 41
 Year                 EventName Driver  BestQualiSeconds  BestQualiLapSeconds    Diff
 2023  Saudi Arabian Grand Prix    SAR           128.510               89.721 -38.789
 2024      São Paulo Grand Prix    ALO            88.998               85.035  -3.963
 2023        Belgian Grand Prix    RIC           122.159              119.483  -2.676
 2025  United States Grand Prix    ALB            94.690               93.670  -1.020
 2025 Emilia Romagna Grand Prix    BEA            76.918               76.077  -0.841
 2022       Austrian Grand Prix    PER            66.143               65.404  -0.739
 2022    Mexico City Grand Prix    MSC            80.419               79.695  -0.724
 2025  United States Grand Prix    STR            94.540               93.878  -0.662
 2024     Australian Grand Prix    RIC            78.085               77.466  -0.619
 2023      

In [5]:
ref = check.copy()

# 1st choice: official results-table time
ref["QualiRef"] = ref["BestQualiSeconds"]
ref["QualiRefSource"] = np.where(ref["QualiRef"].notna(), "results", None)

# 2nd choice: best valid lap from the qualifying lap data (fixes 2025 Miami)
use_lap = ref["QualiRef"].isna() & ref["BestQualiLapSeconds"].notna()
ref.loc[use_lap, "QualiRef"] = ref.loc[use_lap, "BestQualiLapSeconds"]
ref.loc[use_lap, "QualiRefSource"] = "lap_data"

# 3rd choice: team-mate's reference (same car, same session)
teammate_ref = ref.groupby(["Year", "Round", "Team"])["QualiRef"].transform("min")
use_tm = ref["QualiRef"].isna() & teammate_ref.notna()
ref.loc[use_tm, "QualiRef"] = teammate_ref[use_tm]
ref.loc[use_tm, "QualiRefSource"] = "teammate"

print("Where each driver's qualifying reference comes from:")
print(ref["QualiRefSource"].value_counts(dropna=False).to_string())
print("\nStill missing:")
print(ref.loc[ref["QualiRef"].isna(), ["Year", "EventName", "Driver", "Team"]].to_string(index=False))

# Attach to every lap (new columns), checking no rows are gained or lost
n_before = len(laps)
laps = laps.merge(ref[["Year", "Round", "Driver", "QualiRef", "QualiRefSource"]],
                  on=["Year", "Round", "Driver"], how="left")
assert len(laps) == n_before, "Merge changed the number of laps!"
print(f"\n✅ QualiRef attached to {laps['QualiRef'].notna().mean():.1%} of laps")

Where each driver's qualifying reference comes from:
QualiRefSource
results     1789
lap_data      31
teammate       7

Still missing:
Empty DataFrame
Columns: [Year, EventName, Driver, Team]
Index: []

✅ QualiRef attached to 100.0% of laps


In [6]:
# Wet race: any race laps on intermediate or wet tyres
race_flags = (
    laps.groupby(["Year", "Round", "EventName", "Circuit"])["Compound"]
        .apply(lambda c: c.isin(WET).mean())
        .rename("RaceWetTyreShare").reset_index()
)

# Wet qualifying: any qualifying laps on intermediate or wet tyres (rain alone doesn't count)
session = quali.drop_duplicates(["Year", "Round"])[["Year", "Round", "QualiWetTyreShare", "QualiRainfall"]]
race_flags = race_flags.merge(session, on=["Year", "Round"], how="left")

race_flags["WetRace"] = race_flags["RaceWetTyreShare"] > 0
race_flags["WetQuali"] = race_flags["QualiWetTyreShare"] > 0
race_flags["ExcludeRace"] = race_flags["WetRace"] | race_flags["WetQuali"]

print(f"Wet race:   {race_flags['WetRace'].sum()}")
print(f"Wet quali:  {race_flags['WetQuali'].sum()}")
print(f"Both:       {(race_flags['WetRace'] & race_flags['WetQuali']).sum()}")
print(f"Excluded:   {race_flags['ExcludeRace'].sum()} of {len(race_flags)}")

print("\nRaces per season (total / excluded / kept):")
season = race_flags.groupby("Year").agg(Total=("ExcludeRace", "size"), Excluded=("ExcludeRace", "sum"))
season["Kept"] = season["Total"] - season["Excluded"]
print(season.to_string())

print("\nExcluded races and why:")
excl = race_flags[race_flags["ExcludeRace"]].copy()
excl["Reason"] = np.select(
    [excl["WetRace"] & excl["WetQuali"], excl["WetRace"]],
    ["wet race + wet quali", "wet race"], default="wet quali",
)
print(excl[["Year", "EventName", "Reason"]].to_string(index=False))

# --- Cross-check only (not used as a rule): median race-vs-quali gap on rough clean laps ---
rough = laps[
    laps["Compound"].isin(DRY) & laps["PitInTime"].isna() & laps["PitOutTime"].isna()
    & (laps["LapNumber"] > 1) & laps["IsAccurate"].eq(True)
    & laps["TrackStatus"].astype(str).eq("1") & laps["QualiRef"].notna()
]
median_gap = (
    (rough["LapTimeSeconds"] - rough["QualiRef"])
    .groupby([rough["Year"], rough["Round"]]).median()
    .rename("MedianGap").reset_index()
)
race_flags = race_flags.merge(median_gap, on=["Year", "Round"], how="left")

print("\nCross-check — median gap in races flagged for wet qualifying:")
print(race_flags.loc[race_flags["WetQuali"], ["Year", "EventName", "MedianGap"]]
      .sort_values("MedianGap").round(2).to_string(index=False))
print("\nCross-check — lowest median gaps among races we KEEP:")
print(race_flags.loc[~race_flags["ExcludeRace"], ["Year", "EventName", "QualiRainfall", "MedianGap"]]
      .nsmallest(8, "MedianGap").round(2).to_string(index=False))

Wet race:   12
Wet quali:  12
Both:       5
Excluded:   19 of 92

Races per season (total / excluded / kept):
      Total  Excluded  Kept
Year                       
2022     22         7    15
2023     22         4    18
2024     24         4    20
2025     24         4    20

Excluded races and why:
 Year                 EventName               Reason
 2022 Emilia Romagna Grand Prix wet race + wet quali
 2022         Monaco Grand Prix             wet race
 2022       Canadian Grand Prix            wet quali
 2022        British Grand Prix            wet quali
 2022      Singapore Grand Prix wet race + wet quali
 2022       Japanese Grand Prix             wet race
 2022      São Paulo Grand Prix            wet quali
 2023         Monaco Grand Prix             wet race
 2023       Canadian Grand Prix            wet quali
 2023        Belgian Grand Prix            wet quali
 2023          Dutch Grand Prix wet race + wet quali
 2024       Canadian Grand Prix             wet race
 2024   

In [7]:
# F1's own 107% rule: a qualifying time slower than 107% of the fastest is not representative.
pole = ref.groupby(["Year", "Round"])["QualiRef"].transform("min")
too_slow = ref["QualiRef"] > 1.07 * pole

print(f"References slower than 107% of the fastest: {too_slow.sum()}")
print(ref.loc[too_slow, ["Year", "EventName", "Driver", "QualiRef", "QualiRefSource", "BestQualiLapSeconds"]]
      .round(3).to_string(index=False))

# Fix: use the lap-data best if that one is within 107%; otherwise the team-mate's reference
lap_ok = ref["BestQualiLapSeconds"] <= 1.07 * pole
fix_lap = too_slow & lap_ok
ref.loc[fix_lap, "QualiRef"] = ref.loc[fix_lap, "BestQualiLapSeconds"]
ref.loc[fix_lap, "QualiRefSource"] = "lap_data_107"

still_slow = ref["QualiRef"] > 1.07 * pole
tm_ok = ref.loc[~still_slow].groupby(["Year", "Round", "Team"])["QualiRef"].min()
tm_vals = ref.set_index(["Year", "Round", "Team"]).index.map(tm_ok)
fix_tm = still_slow & pd.Series(tm_vals, index=ref.index).notna()
ref.loc[fix_tm, "QualiRef"] = pd.Series(tm_vals, index=ref.index)[fix_tm]
ref.loc[fix_tm, "QualiRefSource"] = "teammate_107"

print(f"\nFixed with lap data: {fix_lap.sum()} | fixed with team-mate: {fix_tm.sum()} | "
      f"still >107%: {(ref['QualiRef'] > 1.07 * pole).sum()}")
print(ref["QualiRefSource"].value_counts().to_string())

# Re-attach the corrected references to every lap
n_before = len(laps)
laps = laps.drop(columns=["QualiRef", "QualiRefSource"]).merge(
    ref[["Year", "Round", "Driver", "QualiRef", "QualiRefSource"]],
    on=["Year", "Round", "Driver"], how="left",
)
assert len(laps) == n_before, "Merge changed the number of laps!"
print("✅ Corrected references attached")

References slower than 107% of the fastest: 54
 Year                 EventName Driver  QualiRef QualiRefSource  BestQualiLapSeconds
 2022     Australian Grand Prix    STR   106.833       lap_data              106.833
 2022 Emilia Romagna Grand Prix    ALB   123.171       lap_data              123.171
 2022       Canadian Grand Prix    GAS    94.492        results               94.492
 2022       Canadian Grand Prix    PER    93.127        results               93.127
 2022       Canadian Grand Prix    LEC    93.008        results               93.008
 2022       Canadian Grand Prix    STR    95.532        results               95.532
 2022       Canadian Grand Prix    TSU    96.575        results               96.575
 2022       Canadian Grand Prix    NOR    94.066        results               94.066
 2022       Canadian Grand Prix    VET    94.512        results               94.512
 2022       Canadian Grand Prix    LAT    95.660        results               95.660
 2022      Singapo

In [8]:
# Race-level flags onto every lap
laps = laps.merge(race_flags[["Year", "Round", "WetRace", "WetQuali", "ExcludeRace"]],
                  on=["Year", "Round"], how="left")

# Track-status flag columns (TrackStatus unchanged)
status = laps["TrackStatus"].astype(str)
laps["IsSafetyCar"] = status.str.contains("4")
laps["IsVSC"] = status.str.contains("6") | status.str.contains("7")
laps["IsRedFlag"] = status.str.contains("5")
laps["IsYellow"] = status.str.contains("2")
laps["IsNeutralised"] = laps["IsSafetyCar"] | laps["IsVSC"] | laps["IsRedFlag"]

# Restart lap: the first green lap after a neutralised lap (tyres and brakes are cold)
laps = laps.sort_values(["Year", "Round", "Driver", "LapNumber"]).reset_index(drop=True)
prev_neutral = laps.groupby(["Year", "Round", "Driver"])["IsNeutralised"].shift(1, fill_value=False)
laps["IsRestartLap"] = prev_neutral & ~laps["IsNeutralised"]

# Retirement: a car about to fail can lose pace, so drop its last 2 laps
print("Most common finishing statuses:")
print(laps.drop_duplicates(["Year", "Round", "Driver"])["Status"].value_counts().head(12).to_string())
finished = laps["Status"].fillna("").str.match(r"^(Finished|Lapped|\+\d+ Laps?)$")
last_lap = laps.groupby(["Year", "Round", "Driver"])["LapNumber"].transform("max")
laps["IsRetirementLap"] = ~finished & (laps["LapNumber"] >= last_lap - 1)

rules = {
    "Wet race or wet quali":         laps["ExcludeRace"].fillna(False),
    "Lap 1 (standing start)":        laps["LapNumber"] <= 1,
    "Pit in-lap or out-lap":         laps["PitInTime"].notna() | laps["PitOutTime"].notna(),
    "SC / VSC / red-flag lap":       laps["IsNeutralised"],
    "Restart lap":                   laps["IsRestartLap"],
    "Not a dry compound":            ~laps["Compound"].isin(DRY),
    "Bad or missing timing":         ~laps["IsAccurate"].eq(True) | laps["FastF1Generated"].eq(True)
                                     | laps["LapTimeSeconds"].isna(),
    "Last 2 laps before retirement": laps["IsRetirementLap"],
}

keep = pd.Series(True, index=laps.index)
rows = []
for name, mask in rules.items():
    removed = int((keep & mask).sum())
    keep &= ~mask
    rows.append((name, removed, int(keep.sum())))

waterfall = pd.DataFrame(rows, columns=["Rule (applied in order)", "Laps removed", "Laps left"])
print(f"\nStarting laps: {len(laps):,}")
print(waterfall.to_string(index=False))
laps["PassesBasicRules"] = keep

Most common finishing statuses:
Status
Finished            1185
Lapped               297
Retired              151
+1 Lap                80
                      19
Collision damage      14
Accident              11
Collision             10
Disqualified           9
Engine                 7
+2 Laps                6
Power Unit             5

Starting laps: 101,291
      Rule (applied in order)  Laps removed  Laps left
        Wet race or wet quali         19817      81474
       Lap 1 (standing start)          1450      80024
        Pit in-lap or out-lap          4999      75025
      SC / VSC / red-flag lap          3512      71513
                  Restart lap          1222      70291
           Not a dry compound           545      69746
        Bad or missing timing            71      69675
Last 2 laps before retirement           125      69550


In [9]:
base = laps[laps["PassesBasicRules"]].copy()
base["DeltaToQuali"] = base["LapTimeSeconds"] - base["QualiRef"]

# Compare each lap with the median of the same driver's NON-yellow laps in the same stint
grp = ["Year", "Round", "Driver", "Stint"]
stint_median = base[~base["IsYellow"]].groupby(grp)["DeltaToQuali"].median().rename("StintMedian")
base = base.join(stint_median, on=grp)
base["DevFromStint"] = base["DeltaToQuali"] - base["StintMedian"]

yellow = base[base["IsYellow"]]
normal = base[~base["IsYellow"]]
print(f"Yellow laps: {len(yellow):,} ({len(yellow) / len(base):.1%} of laps passing the basic rules)")
print("\nYellow laps vs the stint's normal laps (s):")
print(yellow["DevFromStint"].describe().round(2).to_string())
print("\nFor comparison — normal laps vs their own stint median (s):")
print(normal["DevFromStint"].describe().round(2).to_string())
print(f"\nShare more than 1 s slower than the stint median: yellow {(yellow['DevFromStint'] > 1).mean():.1%} "
      f"| normal {(normal['DevFromStint'] > 1).mean():.1%}")

Yellow laps: 1,700 (2.4% of laps passing the basic rules)

Yellow laps vs the stint's normal laps (s):
count    1695.00
mean        0.49
std         1.78
min        -3.38
25%        -0.26
50%         0.17
75%         0.77
max        27.32

For comparison — normal laps vs their own stint median (s):
count    67850.00
mean         0.08
std          0.94
min         -5.42
25%         -0.25
50%          0.00
75%          0.30
max         56.50

Share more than 1 s slower than the stint median: yellow 19.5% | normal 7.0%


In [10]:
# Same 107% idea: a race lap slower than 107% of that race's median clean lap is an incident, not pace
race_median = base.groupby(["Year", "Round"])["LapTimeSeconds"].transform("median")
base["IsSlowOutlier"] = base["LapTimeSeconds"] > 1.07 * race_median

print(f"Slow outliers (>107% of race median): {base['IsSlowOutlier'].sum():,} "
      f"({base['IsSlowOutlier'].mean():.2%})")
print(base.loc[base["IsSlowOutlier"], "DeltaToQuali"].describe().round(2).to_string())

after = base[~base["IsSlowOutlier"]]
print(f"\nLaps left after outliers: {len(after):,}")
print(f"Laps still faster than the driver's qualifying lap: {(after['DeltaToQuali'] < 0).sum()}")
print(after[after["DeltaToQuali"] < 0].groupby(["Year", "EventName", "Driver"]).size()
      .sort_values(ascending=False).head(10).to_string())

print("\nDeltaToQuali after cleaning (s):")
print(after["DeltaToQuali"].describe().round(2).to_string())
print("\nLaps per season:")
print(after.groupby("Year").size().to_string())

Slow outliers (>107% of race median): 80 (0.12%)
count    80.00
mean     21.54
std      14.82
min      10.17
25%      12.24
50%      14.07
75%      22.09
max      62.51

Laps left after outliers: 69,470
Laps still faster than the driver's qualifying lap: 17
Year  EventName         Driver
2023  Qatar Grand Prix  ZHO       17

DeltaToQuali after cleaning (s):
count    69470.00
mean         6.17
std          1.68
min         -1.11
25%          5.03
50%          6.07
75%          7.23
max         14.83

Laps per season:
Year
2022    13326
2023    16619
2024    19745
2025    19780
